# OOD Histopathology — Full Pipeline

DINOv2 baseline → Phikon + CORAL + HED augmentation + TTA

In [ ]:
!pip install -q scikit-image transformers torchstain

## Part 1 — Imports & configuration

In [2]:
import h5py, torch, torch.nn as nn, random, os, gc, time, hashlib, copy
import numpy as np, pandas as pd, torchmetrics
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from skimage.color import rgb2hed, hed2rgb
from transformers import ViTModel
from torchstain.torch.normalizers.macenko import TorchMacenkoNormalizer

SEED = 0
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

## Part 2 — Configuration

In [ ]:
TRAIN_PATH = 'train.h5'
VAL_PATH   = 'val.h5'
TEST_PATH  = 'test.h5'

BATCH_SIZE    = 32
NUM_AUG       = 5       
TTA_N         = 3       
LR            = 5e-4
CORAL_LAMBDA  = 1.0
NUM_EPOCHS    = 100
PATIENCE      = 20
HIDDEN_DIM    = 128
SAVE_DIR      = 'weights'
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print(f'Device: {device}')


Device: mps


## Part 3 — Data loading

In [ ]:
class H5Dataset(Dataset):
    """
    Loads an entire .h5 file into RAM.
    Auto-detects whether pixel values are in [0,1] or [0,255].
    In 'test' mode labels and centers are set to sentinel values (-1, 0).
    """
    def __init__(self, path, mode='train'):
        self.mode = mode
        self.imgs, self.labels, self.centers, self.img_ids = [], [], [], []
        print(f'Loading {path} ...', end=' ', flush=True)
        with h5py.File(path, 'r') as hdf:
            for k in list(hdf.keys()):
                self.imgs.append(np.array(hdf[k]['img']))
                self.img_ids.append(int(k))
                if mode == 'train':
                    self.labels.append(int(np.array(hdf[k]['label'])))
                    self.centers.append(int(np.array(hdf[k]['metadata'])[0]))
                else:
                    self.labels.append(-1)
                    self.centers.append(0)
        self.imgs = np.stack(self.imgs)
        self.vmax = float(self.imgs.max())
        self.needs_rescale = self.vmax > 1.5
        print(f'{len(self.imgs)} images | '
              f'range [{self.imgs.min():.1f}, {self.vmax:.1f}] | '
              f'rescale={self.needs_rescale}')

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.imgs[idx]).float()
        if self.needs_rescale:
            img = img / 255.0
        return img, self.labels[idx], self.centers[idx], self.img_ids[idx]


class PrecomputedDataset(Dataset):
    """Wraps pre-extracted embeddings for the lightweight head training loop."""
    def __init__(self, features, labels, centers):
        self.features = features
        self.labels   = labels.unsqueeze(-1).float()
        self.centers  = centers

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], self.centers[idx]


## Part 4 — Augmentation & stain normalisation

In [ ]:
def stain_augment(img_np, sigma=0.2):
    """
    Randomly perturb each HED stain channel independently.
    img_np: float32 [H,W,3] in [0,1].
    """
    hed = rgb2hed(img_np)
    for ch in range(3):
        hed[:, :, ch] = (hed[:, :, ch] * np.random.uniform(1 - sigma, 1 + sigma)
                         + np.random.uniform(-sigma, sigma))
    return np.clip(hed2rgb(hed), 0, 1).astype(np.float32)


_cj  = transforms.ColorJitter(brightness=0.3, contrast=0.3,
                               saturation=0.3, hue=0.1)
_geo = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
])
_tta = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
])


def augment_image(img_tensor):
    """Full training augmentation: HED stain jitter + geometric + colour jitter."""
    img_np = np.clip(img_tensor.permute(1, 2, 0).numpy(), 0, 1).astype(np.float32)
    img_np = stain_augment(img_np)
    t = torch.from_numpy(img_np).permute(2, 0, 1)
    t = (t * 255).clamp(0, 255).to(torch.uint8)
    return _geo(_cj(t)).float() / 255.0


print('Fitting Macenko stain normaliser ...')
macenko = TorchMacenkoNormalizer()
with h5py.File(TRAIN_PATH, 'r') as hdf:
    ref_key = list(hdf.keys())[0]
    ref_img = torch.tensor(np.array(hdf[ref_key]['img']))
    ref_u8  = ((ref_img * 255).clamp(0, 255).byte()
               if ref_img.max() <= 1.5 else ref_img.byte())
macenko.fit(ref_u8)
print('Macenko normaliser ready.')


def apply_macenko(img_tensor):
    """Normalise a single [C,H,W] float tensor. Returns [C,H,W] float."""
    try:
        u8 = (img_tensor * 255).clamp(0, 255).byte()
        norm, _, _ = macenko.normalize(u8, stains=False)   
        return norm.permute(2, 0, 1).float() / 255.0
    except Exception:
        return img_tensor   


Fitting Macenko stain normaliser ...
Macenko normaliser ready.


## Part 5 — Model definitions

In [ ]:
def coral_loss(features, centers):
    """
    Deep CORAL: average pairwise Frobenius distance between per-centre
    covariance matrices, normalised by 4d².
    """
    unique = torch.unique(centers)
    if len(unique) < 2:
        return torch.tensor(0.0, device=features.device)
    loss, n = torch.tensor(0.0, device=features.device), 0
    for i in range(len(unique)):
        for j in range(i + 1, len(unique)):
            s = features[centers == unique[i]]
            t = features[centers == unique[j]]
            if len(s) < 2 or len(t) < 2:
                continue
            d = s.size(1)
            loss += (torch.norm(torch.cov(s.T) - torch.cov(t.T), p='fro') ** 2
                     / (4 * d * d))
            n += 1
    return loss / max(n, 1)


class LinearProbe(nn.Module):
    """Single Linear + Sigmoid. Returns (pred, features) for compatibility."""
    def __init__(self, feat_dim, **_):
        super().__init__()
        self.fc      = nn.Linear(feat_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.fc(x)), x


class MLPHead(nn.Module):
    """Two-layer MLP. The hidden layer h is returned for CORAL regularisation."""
    def __init__(self, feat_dim, hidden=128, **_):
        super().__init__()
        self.fc1     = nn.Linear(feat_dim, hidden)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(hidden, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h = self.relu(self.fc1(x))
        return self.sigmoid(self.fc2(h)), h


class DINOv2Wrapper(nn.Module):
    """
    Wraps a DINOv2 model to expose the same .last_hidden_state interface
    as HuggingFace ViTModel, so both backbones share one precompute path.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        cls = self.model(pixel_values)          # (B, D)
        class _Out: pass
        out = _Out()
        out.last_hidden_state = cls.unsqueeze(1)  # (B, 1, D)
        return out


## Part 6 — Backbone loading

In [ ]:
print('Loading DINOv2-ViT-S/14 ...')
_dinov2_raw  = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
_dinov2_raw.eval()
dinov2       = DINOv2Wrapper(_dinov2_raw)
dinov2_dim   = _dinov2_raw.num_features        # 384

dinov2_transform = transforms.Compose([
    transforms.Resize((98, 98)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Loading Phikon ...')
phikon = ViTModel.from_pretrained('owkin/phikon',
                                   add_pooling_layer=False).float().to(device)
phikon.eval()
phikon_dim = phikon.config.hidden_size          # 768

n_gpus = torch.mps.device_count()
if n_gpus > 1:
    phikon = nn.DataParallel(phikon)
    BATCH_SIZE *= n_gpus
    print(f'DataParallel ON — effective batch size: {BATCH_SIZE}')

phikon_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Quick sanity-check forward passes
with torch.no_grad():
    _ = dinov2(pixel_values=torch.randn(1, 3, 98,  98,  device=device))
    _ = phikon (pixel_values=torch.randn(1, 3, 224, 224, device=device))
print('Both backbones OK.')


Loading DINOv2-ViT-S/14 ...


Using cache found in /Users/khlil/.cache/torch/hub/facebookresearch_dinov2_main
/Users/khlil/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/khlil/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/khlil/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Loading Phikon ...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: owkin/phikon
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Both backbones OK.


## Part 7 — Feature precomputation

In [ ]:
@torch.no_grad()
def precompute(path, mode, backbone, transform,
               num_aug=1, use_macenko=False, cache_key=None):
    """
    Extract CLS-token embeddings for every image in an .h5 file.

    path         : path to .h5 file
    mode         : 'train' or 'test'
    backbone     : DINOv2Wrapper or ViTModel (both return .last_hidden_state)
    transform    : resize + normalise transform
    num_aug      : 1 = no aug; N = 1 clean pass + (N-1) HED-augmented passes
    use_macenko  : apply Macenko normalisation before transform
    cache_key    : if given, cache/load from SAVE_DIR/<cache_key>.pt
    """
    cache_path = os.path.join(SAVE_DIR, f'{cache_key}.pt') if cache_key else None
    if cache_path and os.path.exists(cache_path):
        print(f'  [cache] loading {cache_path}')
        c = torch.load(cache_path, weights_only=True)
        return c['f'], c['l'], c['c']

    ds = H5Dataset(path, mode)
    dl = DataLoader(ds, shuffle=False, batch_size=BATCH_SIZE, num_workers=0)
    t0 = time.time()

    all_f, all_l, all_c = [], [], []

    for aug_i in range(num_aug):
        desc = f'Extracting aug={aug_i}/{num_aug - 1}'
        for imgs, labels, centers, _ in tqdm(dl, desc=desc, leave=False):
            batch = []
            for img in imgs:
                if aug_i > 0:
                    img = augment_image(img)        # HED + geo aug
                if use_macenko:
                    img = apply_macenko(img)
                batch.append(transform(img))
            batch = torch.stack(batch).to(device)
            out   = backbone(pixel_values=batch)
            all_f.append(out.last_hidden_state[:, 0, :].cpu())
            all_l.append(labels)
            all_c.append(centers)

    feats   = torch.cat(all_f)
    labels_ = torch.cat(all_l)
    centers_= torch.cat(all_c)

    print(f'  {feats.shape} in {time.time() - t0:.0f}s')

    if cache_path:
        torch.save({'f': feats, 'l': labels_, 'c': centers_}, cache_path)
        print(f'  [cache] saved {cache_path}')

    del ds; gc.collect(); torch.cuda.empty_cache()
    return feats, labels_, centers_


## Part 8 — Training loop

In [ ]:
def train_head(tf, tl, tc, vf, vl, vc,
               head_cls, feat_dim,
               coral_lam=0.0, hidden=HIDDEN_DIM,
               lr=LR, bs=BATCH_SIZE,
               epochs=NUM_EPOCHS, patience=PATIENCE,
               save_path=None):
    """
    Train a classification head on pre-extracted features.

    head_cls  : LinearProbe or MLPHead
    coral_lam : weight for CORAL loss (0 = pure BCE)
    save_path : where to save best weights; auto-generated if None
    Returns   : (head, best_val_acc)
    """
    if save_path is None:
        save_path = os.path.join(SAVE_DIR, 'best_head.pth')

    tloader = DataLoader(PrecomputedDataset(tf, tl, tc),
                         shuffle=True,  batch_size=bs, drop_last=False)
    vloader = DataLoader(PrecomputedDataset(vf, vl, vc),
                         shuffle=False, batch_size=bs)

    head  = head_cls(feat_dim=feat_dim, hidden=hidden).to(device)
    opt   = torch.optim.Adam(head.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.BCELoss()
    met   = torchmetrics.Accuracy('binary')

    best_acc, best_ep = 0.0, 0

    for ep in range(epochs):
        head.train()
        tL, tA, tC = [], [], []
        for f, l, c in tloader:
            f, l = f.to(device), l.to(device)
            opt.zero_grad()
            p, h = head(f)
            bce  = crit(p, l)
            cl   = coral_loss(h, c.to(device)) if coral_lam > 0 else torch.tensor(0.)
            (bce + coral_lam * cl).backward()
            opt.step()
            tL.append(bce.item()); tC.append(cl.item())
            tA.append(met(p.cpu(), l.int().cpu()).item())
        sched.step()

        head.eval()
        vA = []
        with torch.no_grad():
            for f, l, _ in vloader:
                p, _ = head(f.to(device))
                vA.append(met(p.cpu(), l.int().cpu()).item())

        va      = np.mean(vA)
        cur_lr  = sched.get_last_lr()[0]
        coral_s = f' coral={np.mean(tC):.4f}' if coral_lam > 0 else ''
        print(f'Ep [{ep+1:3d}/{epochs}] '
              f'train loss={np.mean(tL):.4f} acc={np.mean(tA):.4f}{coral_s} | '
              f'val acc={va:.4f} | lr={cur_lr:.1e}')

        if va > best_acc:
            print(f'  → New best: {best_acc:.4f} → {va:.4f}')
            best_acc, best_ep = va, ep
            torch.save(head.state_dict(), save_path)

        if ep - best_ep >= patience:
            print(f'Early stop at epoch {ep + 1}')
            break

    head.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'Best epoch: {best_ep + 1} | best val acc: {best_acc:.4f}')
    return head, best_acc


## Part 9 — Main training run (Phikon + MLP + CORAL + HED aug)

In [17]:
print('=== Precomputing Phikon features ===')
key_tr = hashlib.md5(f'phikon_{NUM_AUG}_{SEED}'.encode()).hexdigest()[:8]
key_vl = hashlib.md5(f'phikon_val_1_{SEED}'.encode()).hexdigest()[:8]

print('Train:')
tf, tl, tc = precompute(TRAIN_PATH, 'train', phikon, phikon_transform,
                        num_aug=NUM_AUG, use_macenko=False,
                        cache_key=f'phikon_train_{key_tr}')
print('Val:')
vf, vl, vc = precompute(VAL_PATH, 'train', phikon, phikon_transform,
                        num_aug=1, use_macenko=False,
                        cache_key=f'phikon_val_{key_vl}')
print(f'Train: {tf.shape}  Val: {vf.shape}')

=== Precomputing Phikon features ===
Train:
  [cache] loading weights/phikon_train_a467a31a.pt
Val:
  [cache] loading weights/phikon_val_0434982e.pt
Train: torch.Size([500000, 768])  Val: torch.Size([34904, 768])


In [ ]:
print('=== Training Phikon + MLP + CORAL ===')
final_head, final_val_acc = train_head(
    tf, tl, tc, vf, vl, vc,
    head_cls  = MLPHead,
    feat_dim  = phikon_dim,
    coral_lam = CORAL_LAMBDA,
    save_path = os.path.join(SAVE_DIR, 'phikon_mlp_coral_final.pth'),
)
print(f'Final model saved. Val accuracy: {final_val_acc * 100:.2f}%')


=== Training Phikon + MLP + CORAL ===
Ep [  1/100] train loss=0.2841 acc=0.8578 coral=0.0071 | val acc=0.9565 | lr=5.0e-04
  → New best: 0.0000 → 0.9565
Ep [  2/100] train loss=0.2586 acc=0.8725 coral=0.0054 | val acc=0.9562 | lr=5.0e-04
Ep [  3/100] train loss=0.2509 acc=0.8768 coral=0.0053 | val acc=0.9613 | lr=5.0e-04
  → New best: 0.9565 → 0.9613
Ep [  4/100] train loss=0.2454 acc=0.8791 coral=0.0054 | val acc=0.9584 | lr=5.0e-04
Ep [  5/100] train loss=0.2422 acc=0.8812 coral=0.0057 | val acc=0.9514 | lr=5.0e-04
Ep [  6/100] train loss=0.2394 acc=0.8826 coral=0.0058 | val acc=0.9573 | lr=5.0e-04
Ep [  7/100] train loss=0.2376 acc=0.8835 coral=0.0059 | val acc=0.9546 | lr=4.9e-04
Ep [  8/100] train loss=0.2350 acc=0.8846 coral=0.0060 | val acc=0.9557 | lr=4.9e-04
Ep [  9/100] train loss=0.2330 acc=0.8860 coral=0.0061 | val acc=0.9534 | lr=4.9e-04
Ep [ 10/100] train loss=0.2322 acc=0.8865 coral=0.0062 | val acc=0.9462 | lr=4.9e-04
Ep [ 11/100] train loss=0.2306 acc=0.8871 coral=0.00

## Part 10 — Test-set prediction with TTA

In [ ]:
@torch.no_grad()
def predict_tta(backbone, transform, head, test_path,
                tta_n=TTA_N, bs=BATCH_SIZE,
                save_path='submission.csv'):
    """
    Predict on test set with TTA.
    tta_n=1 means a single clean pass (no augmentation).
    """
    head.eval(); backbone.eval()
    ds = H5Dataset(test_path, 'test')
    dl = DataLoader(ds, shuffle=False, batch_size=bs, num_workers=0)
    t0 = time.time()

    all_p, all_id = [], []
    for imgs, _, _, ids in tqdm(dl, desc='Predicting'):
        batch_probs = []
        for ti in range(tta_n):
            batch = []
            for img in imgs:
                proc = (img if ti == 0
                        else _tta((img * 255).clamp(0, 255)
                                  .to(torch.uint8)).float() / 255.0)
                batch.append(transform(proc))
            batch = torch.stack(batch).to(device)
            out   = backbone(pixel_values=batch)
            f     = out.last_hidden_state[:, 0, :]
            p, _  = head(f)
            batch_probs.append(p.cpu())

        avg = torch.stack(batch_probs).mean(0).squeeze(-1)
        all_p.append((avg > 0.5).int())
        all_id.append(ids)

    del ds; gc.collect()
    ids_arr   = torch.cat(all_id).numpy()
    preds_arr = torch.cat(all_p).numpy()
    print(f'Done in {time.time() - t0:.0f}s')

    df = pd.DataFrame({'ID': ids_arr, 'Pred': preds_arr}).set_index('ID')
    df.to_csv(save_path)
    print(f'Saved {save_path} ({len(df)} predictions)')
    dist = df['Pred'].value_counts().to_dict()
    print(f'Class distribution: {dist}')
    return df


del tf, tl, tc, vf, vl, vc; gc.collect(); torch.mps.empty_cache()

final_head = MLPHead(feat_dim=phikon_dim, hidden=HIDDEN_DIM).to(device)
final_head.load_state_dict(
    torch.load(os.path.join(SAVE_DIR, 'ablation_Phikon_p_MLP_p_CORAL_(λ=1.0).pth'),
               weights_only=True))

submission = predict_tta(phikon, phikon_transform, final_head,
                         TEST_PATH, tta_n=1,
                         save_path='submission_final.csv')

Loading test.h5 ... 85054 images | range [0.0, 1.0] | rescale=False


Predicting:   0%|          | 0/2658 [00:00<?, ?it/s]

Done in 816s
Saved submission_final.csv (85054 predictions)
Class distribution: {0: 47224, 1: 37830}


## Part 11 — Ablation study (Table 1)

Runs every configuration from the report table. Each row trains a fresh head.
Results are printed as a formatted table and saved to `ablation_results.csv`.

In [ ]:
ABLATION_EPOCHS  = 60
ABLATION_PATIENCE= 15
ABLATION_BS      = 32
ABLATION_LR      = 5e-4


ABLATION_ROWS = [
    dict(name='DINOv2 + linear probe (baseline)',
         backbone='dinov2', num_aug=1,       use_macenko=False,
         head_cls=LinearProbe, coral_lam=0.0, tta_n=0),

    dict(name='  + Macenko norm + TTA (8 views)',
         backbone='dinov2', num_aug=1,       use_macenko=True,
         head_cls=LinearProbe, coral_lam=0.0, tta_n=8),

    dict(name='Phikon + linear probe',
         backbone='phikon', num_aug=1,       use_macenko=False,
         head_cls=LinearProbe, coral_lam=0.0, tta_n=0),

    dict(name='Phikon + MLP (no CORAL)',
         backbone='phikon', num_aug=1,       use_macenko=False,
         head_cls=MLPHead,    coral_lam=0.0, tta_n=0),

    dict(name='Phikon + MLP + CORAL (λ=1.0)',
         backbone='phikon', num_aug=1,       use_macenko=False,
         head_cls=MLPHead,    coral_lam=1.0, tta_n=0),

    dict(name='  + HED augmentation (×5)',
         backbone='phikon', num_aug=5,       use_macenko=False,
         head_cls=MLPHead,    coral_lam=1.0, tta_n=0),

    dict(name='  + TTA (3 views)  [final]',
         backbone='phikon', num_aug=5,       use_macenko=False,
         head_cls=MLPHead,    coral_lam=1.0, tta_n=3),
]

In [ ]:
def eval_with_tta(head, vf, vl, tta_n, bs=ABLATION_BS):
    """
    Re-evaluate a trained head with TTA on already-extracted val features.
    Since features are fixed, 'TTA' here averages tta_n stochastic forward
    passes (head temporarily put in train mode to allow any future dropout).
    For a rigorous image-level TTA the backbone would need to be re-run with
    different augmentations — done in predict_tta() for the final submission.
    """
    met = torchmetrics.Accuracy('binary')
    passes = []
    for _ in range(max(tta_n, 1)):
        head.train()   
        preds = []
        with torch.no_grad():
            for i in range(0, len(vf), bs):
                p, _ = head(vf[i:i + bs].to(device))
                preds.append(p.cpu())
        passes.append(torch.cat(preds))
    head.eval()
    avg = torch.stack(passes).mean(0).squeeze(-1) 
    return met(avg, vl.squeeze(-1).int()).item() * 100



RESUME_FROM = '  + Macenko norm + TTA (8 views)'  # To avoid rerunning every ablation at each run. 

if RESUME_FROM is None:
    results = []
elif 'results' not in dir():
    results = []   

if '_feat_cache' not in dir():
    _feat_cache: dict = {}

_skipping = RESUME_FROM is not None  

for row in ABLATION_ROWS:
    if _skipping:
        if row['name'] == RESUME_FROM:
            _skipping = False  
        else:
            row_id = row['name'].strip().replace(' ', '_').replace('+', 'p')
            sp = os.path.join(SAVE_DIR, f'ablation_{row_id[:40]}.pth')
            if os.path.exists(sp):
                dim      = dinov2_dim if row['backbone'] == 'dinov2' else phikon_dim
                head_tmp = row['head_cls'](feat_dim=dim, hidden=HIDDEN_DIM).to(device)
                head_tmp.load_state_dict(torch.load(sp, weights_only=True))
                head_tmp.eval()

                bname    = row['backbone']
                feat_key = (bname, row['num_aug'], row['use_macenko'])
                if feat_key not in _feat_cache:
                    bb = dinov2 if bname == 'dinov2' else phikon
                    tr = dinov2_transform if bname == 'dinov2' else phikon_transform
                    ck_vl = f'{bname}_aug1_mac{int(row["use_macenko"])}_val'
                    _, _vl_tmp, _vc_tmp = precompute(
                        VAL_PATH, 'train', bb, tr,
                        num_aug=1, use_macenko=row['use_macenko'],
                        cache_key=ck_vl,
                    )
                    # We only need vf here; store a minimal entry
                    _feat_cache[feat_key] = (None, None, None,
                                             None, _vl_tmp, _vc_tmp)
                _, _, _, _vf_tmp, _vl_tmp, _ = _feat_cache[feat_key]

                if _vf_tmp is None:
                    # vf not yet in cache — load it now
                    bb = dinov2 if bname == 'dinov2' else phikon
                    tr = dinov2_transform if bname == 'dinov2' else phikon_transform
                    ck_vl = f'{bname}_aug1_mac{int(row["use_macenko"])}_val'
                    _vf_tmp, _vl_tmp, _vc_tmp = precompute(
                        VAL_PATH, 'train', bb, tr,
                        num_aug=1, use_macenko=row['use_macenko'],
                        cache_key=ck_vl,
                    )
                    old = _feat_cache[feat_key]
                    _feat_cache[feat_key] = (old[0], old[1], old[2],
                                             _vf_tmp, _vl_tmp, _vc_tmp)

                recovered_acc = eval_with_tta(head_tmp, _vf_tmp, _vl_tmp,
                                              max(row['tta_n'], 1))
                results.append({'Configuration': row['name'],
                                'Val Acc (%)': round(recovered_acc, 2)})
                print(f'  [skipped] {row["name"]}  → {recovered_acc:.2f}% (recovered)')
                del head_tmp
            else:
                print(f'  [skipped] {row["name"]}  (no saved weights found)')
            continue  

    sep = '─' * 62
    print(f'\n{sep}')
    print(f'  {row["name"]}')
    print(sep)

    bname    = row['backbone']
    feat_key = (bname, row['num_aug'], row['use_macenko'])

    if feat_key not in _feat_cache or _feat_cache[feat_key][0] is None:
        bb  = dinov2 if bname == 'dinov2' else phikon
        tr  = dinov2_transform if bname == 'dinov2' else phikon_transform

        ck_tr = f'{bname}_aug{row["num_aug"]}_mac{int(row["use_macenko"])}_train'
        ck_vl = f'{bname}_aug1_mac{int(row["use_macenko"])}_val'

        print('  Extracting train features ...')
        _tf, _tl, _tc = precompute(
            TRAIN_PATH, 'train', bb, tr,
            num_aug=row['num_aug'], use_macenko=row['use_macenko'],
            cache_key=ck_tr,
        )
        print('  Extracting val features ...')
        _vf, _vl, _vc = precompute(
            VAL_PATH, 'train', bb, tr,
            num_aug=1, use_macenko=row['use_macenko'],
            cache_key=ck_vl,
        )
        _feat_cache[feat_key] = (_tf, _tl, _tc, _vf, _vl, _vc)
    else:
        print('  Reusing cached features.')

    _tf, _tl, _tc, _vf, _vl, _vc = _feat_cache[feat_key]
    dim = dinov2_dim if row['backbone'] == 'dinov2' else phikon_dim

    row_id = row['name'].strip().replace(' ', '_').replace('+', 'p')
    sp = os.path.join(SAVE_DIR, f'ablation_{row_id[:40]}.pth')

    head_obj, best_acc = train_head(
        _tf, _tl, _tc, _vf, _vl, _vc,
        head_cls  = row['head_cls'],
        feat_dim  = dim,
        coral_lam = row['coral_lam'],
        lr        = ABLATION_LR,
        bs        = ABLATION_BS,
        epochs    = ABLATION_EPOCHS,
        patience  = ABLATION_PATIENCE,
        save_path = sp,
    )

    val_acc = best_acc * 100
    if row['tta_n'] > 0:
        head_obj.load_state_dict(torch.load(sp, weights_only=True))
        val_acc = eval_with_tta(head_obj, _vf, _vl, row['tta_n'])
        print(f'  After TTA ({row["tta_n"]} views): {val_acc:.2f}%')

    results.append({'Configuration': row['name'],
                    'Val Acc (%)': round(val_acc, 2)})
    print(f'  → Val Acc = {val_acc:.2f}%')
    del head_obj; gc.collect(); torch.cuda.empty_cache()


  [cache] loading weights/dinov2_aug1_mac0_val.pt
  [cache] loading weights/dinov2_aug1_mac0_val.pt
  [skipped] DINOv2 + linear probe (baseline)  → 88.29% (recovered)

──────────────────────────────────────────────────────────────
    + Macenko norm + TTA (8 views)
──────────────────────────────────────────────────────────────
  Extracting train features ...
  [cache] loading weights/dinov2_aug1_mac1_train.pt
  Extracting val features ...
  [cache] loading weights/dinov2_aug1_mac1_val.pt
Ep [  1/60] train loss=0.1920 acc=0.9273 | val acc=0.7575 | lr=5.0e-04
  → New best: 0.0000 → 0.7575
Ep [  2/60] train loss=0.1551 acc=0.9415 | val acc=0.7827 | lr=5.0e-04
  → New best: 0.7575 → 0.7827
Ep [  3/60] train loss=0.1476 acc=0.9448 | val acc=0.8038 | lr=5.0e-04
  → New best: 0.7827 → 0.8038
Ep [  4/60] train loss=0.1439 acc=0.9459 | val acc=0.7721 | lr=4.9e-04
Ep [  5/60] train loss=0.1415 acc=0.9469 | val acc=0.8169 | lr=4.9e-04
  → New best: 0.8038 → 0.8169
Ep [  6/60] train loss=0.1400 ac

Extracting aug=0/0:   0%|          | 0/3125 [00:00<?, ?it/s]

  torch.Size([100000, 768]) in 951s
  [cache] saved weights/phikon_aug1_mac0_train.pt
  Extracting val features ...
Loading val.h5 ... 34904 images | range [0.0, 1.0] | rescale=False


Extracting aug=0/0:   0%|          | 0/1091 [00:00<?, ?it/s]

  torch.Size([34904, 768]) in 336s
  [cache] saved weights/phikon_aug1_mac0_val.pt
Ep [  1/60] train loss=0.0447 acc=0.9851 | val acc=0.9690 | lr=5.0e-04
  → New best: 0.0000 → 0.9690
Ep [  2/60] train loss=0.0292 acc=0.9903 | val acc=0.9691 | lr=5.0e-04
  → New best: 0.9690 → 0.9691
Ep [  3/60] train loss=0.0268 acc=0.9910 | val acc=0.9672 | lr=5.0e-04
Ep [  4/60] train loss=0.0255 acc=0.9916 | val acc=0.9665 | lr=4.9e-04
Ep [  5/60] train loss=0.0245 acc=0.9922 | val acc=0.9589 | lr=4.9e-04
Ep [  6/60] train loss=0.0237 acc=0.9923 | val acc=0.9631 | lr=4.9e-04
Ep [  7/60] train loss=0.0231 acc=0.9926 | val acc=0.9554 | lr=4.8e-04
Ep [  8/60] train loss=0.0229 acc=0.9926 | val acc=0.9680 | lr=4.8e-04
Ep [  9/60] train loss=0.0226 acc=0.9928 | val acc=0.9670 | lr=4.7e-04
Ep [ 10/60] train loss=0.0221 acc=0.9929 | val acc=0.9622 | lr=4.7e-04
Ep [ 11/60] train loss=0.0219 acc=0.9930 | val acc=0.9602 | lr=4.6e-04
Ep [ 12/60] train loss=0.0216 acc=0.9932 | val acc=0.9643 | lr=4.5e-04
Ep [ 

...

Extracting aug=1/4:   0%|          | 0/3125 [00:00<?, ?it/s]

Extracting aug=2/4:   0%|          | 0/3125 [00:00<?, ?it/s]

Extracting aug=3/4:   0%|          | 0/3125 [00:00<?, ?it/s]

Extracting aug=4/4:   0%|          | 0/3125 [00:00<?, ?it/s]

  torch.Size([500000, 768]) in 5359s
  [cache] saved weights/phikon_aug5_mac0_train.pt
  Extracting val features ...
  [cache] loading weights/phikon_aug1_mac0_val.pt
Ep [  1/60] train loss=0.2839 acc=0.8580 coral=0.0068 | val acc=0.9567 | lr=5.0e-04
  → New best: 0.0000 → 0.9567
Ep [  2/60] train loss=0.2583 acc=0.8727 coral=0.0052 | val acc=0.9636 | lr=5.0e-04
  → New best: 0.9567 → 0.9636
Ep [  3/60] train loss=0.2509 acc=0.8770 coral=0.0052 | val acc=0.9502 | lr=5.0e-04
Ep [  4/60] train loss=0.2466 acc=0.8789 coral=0.0054 | val acc=0.9526 | lr=4.9e-04
Ep [  5/60] train loss=0.2428 acc=0.8813 coral=0.0056 | val acc=0.9441 | lr=4.9e-04
Ep [  6/60] train loss=0.2402 acc=0.8822 coral=0.0058 | val acc=0.9555 | lr=4.9e-04
Ep [  7/60] train loss=0.2379 acc=0.8835 coral=0.0058 | val acc=0.9512 | lr=4.8e-04
Ep [  8/60] train loss=0.2363 acc=0.8844 coral=0.0059 | val acc=0.9561 | lr=4.8e-04
Ep [  9/60] train loss=0.2347 acc=0.8851 coral=0.0060 | val acc=0.9552 | lr=4.7e-04
Ep [ 10/60] train

In [12]:
W = 58
print('\n' + '═' * W)
print('  TABLE 1 — Ablation on held-out validation set (centre 1)')
print('═' * W)
print(f'  {"Configuration":<43}  {"Val Acc (%)":>9}')
print('─' * W)
for r in results:
    print(f'  {r["Configuration"]:<43}  {r["Val Acc (%)"]:>9.2f}')
print('═' * W)

ablation_df = pd.DataFrame(results)
ablation_df.to_csv('ablation_results.csv', index=False)
print('\nSaved ablation_results.csv')


══════════════════════════════════════════════════════════
  TABLE 1 — Ablation on held-out validation set (centre 1)
══════════════════════════════════════════════════════════
  Configuration                                Val Acc (%)
──────────────────────────────────────────────────────────
  DINOv2 + linear probe (baseline)                 88.29
    + Macenko norm + TTA (8 views)                 82.35
  Phikon + linear probe                            96.91
  Phikon + MLP (no CORAL)                          97.40
  Phikon + MLP + CORAL (λ=1.0)                     97.32
    + HED augmentation (×5)                        96.36
    + TTA (3 views)  [final]                       95.90
══════════════════════════════════════════════════════════

Saved ablation_results.csv
